# Traveling salesman with native QAOA

All 17 source stages, four cities, 16 assignment bits, eight QAOA layers, 150 evaluations of the lower 60 percent sampled cost, and 2,048 draws for each optimization, final and uniform histogram.

Use native COBYLA and the grouped pi-ramp schedule from the available Classiq SDK reference. Preserve the last evaluated native binding and use independent local seeds for optimized and uniform readout. Enumerate the 24 feasible open paths as the classical reference. Report any invalid observed assignment as sampled.

## Draw the original four-city directed graph

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.logistics.traveling_salesman_problem.overview.model import (
    EDGES,
    NONEDGE,
    make_model,
    visits,
)
from examples.synthesis.applications.logistics.traveling_salesman_problem.overview.plots import (
    save_costs,
    save_graph,
    save_trace,
)
from examples.synthesis.applications.logistics.traveling_salesman_problem.overview.quantum import (
    LAYERS,
    build_program,
)
from examples.synthesis.applications.logistics.traveling_salesman_problem.overview.workflow import (
    MAX_QUERIES,
    QUANTILE,
    SHOTS,
    run_workflow,
)

folder = Path("artifacts/development/traveling-salesman")
display(Image(filename=str(save_graph(folder / "graph.png"))))


## Use the bounded native model

In [ ]:
print("Source directed edges:", EDGES)


## Define one city per slot and one slot per city

In [ ]:
model = make_model()


## Preserve the three directed transitions

In [ ]:
print("Nonedge cost:", NONEDGE)


## Generate the original distance matrix

In [ ]:
print("Distances:", model.distances)


## Keep all eight QAOA layers

In [ ]:
print("Native layers:", LAYERS)


## Synthesize the uniform initial ansatz

In [ ]:
initial = build_program(model, (0.0,) * (2 * LAYERS))
print(initial.report["resources"])


## Run the finite low-tail optimization

In [ ]:
print("Query limit, lower fraction and shots:", MAX_QUERIES, QUANTILE, SHOTS)
result = run_workflow()
print(
    "Actual queries and objective draws:",
    len(result.evaluations),
    len(result.evaluations) * SHOTS,
)
print("Budget stop and convergence:", result.budget_reached, result.converged)


## Plot the actual sampled trace

In [ ]:
display(Image(filename=str(save_trace(result, folder / "trace.png"))))


## Read the optimized sample

In [ ]:
print(
    "Optimized and uniform draws:",
    result.optimized_draw.numel(),
    result.uniform_draw.numel(),
)


## Validate the independent uniform readout

In [ ]:
assert result.uniform_draw.numel() == SHOTS


## Compare both observed cost histograms

In [ ]:
display(Image(filename=str(save_costs(result, folder / "costs.png"))))


## Keep the best observed assignment

In [ ]:
print(
    "Observed word and cost:",
    result.best_word,
    float(result.model.cost[result.best_word]),
)


## Enumerate all feasible open paths

In [ ]:
print("Exact route and cost:", result.exact[1], result.exact[0])


## Read the classical path

In [ ]:
assert len(result.exact[1]) == 4


## Restore the sampled assignment matrix

In [ ]:
print("Observed assignment:", visits(result.best_word))


## Show the route with an explicit feasibility label

In [ ]:
print("Observed route:", result.best_route)
if result.best_route is None:
    label = "Exact path, observed assignment infeasible"
else:
    label = f"Observed path, cost {float(result.model.cost[result.best_word]):g}"
display(
    Image(
        filename=str(
            save_graph(
                folder / "route.png",
                route=result.best_route or result.exact[1],
                title=label,
            )
        )
    )
)
